Before Going into the LCEL, lets understand the What a Chain is:

- a chain is component that is Connecting Multiple Langchain compnents together to perform a task.
- we can think it is a pipeline
```
Input
 ↓
Prompt
 ↓
LLM
 ↓
Parser
 ↓
Output
```
- instead of manually writing: 
```
formatted_prompt = prompt.format(...)
response = llm.invoke(formatted_prompt)
output = parser.invoke(response)
```
- langchain allows us to connect everything as a chain.

##### Why do we need chains?
- without chains , we have to write a lines of code and hard to maintain basically like 
```
formatted_prompt = prompt.invoke(
{
 "topic":"Spark"
}
)

response = llm.invoke(
    formatted_prompt
)

result = parser.invoke(
    response
)
```
- with chains ```chain = prompt | llm | parser``` thats it 

#### LCEL
- lcel stands for ```LangChain Expression language``` it is a decalrative way to connect to components.
- think like 
```
Component 1
    |
Component 2
    |
Component 3
```

- here | means ```pass ouput to next component```

##### Simple chain
- prompt --> Chatmodel --> StringOutput
- code 
```
from langchain_core.outputparsers import StrOutputParser
chain = (prompt | llm | StrOutput())
```

- visual: Prompt --> Chatmodel --> string parser

In [ ]:
# Example :
prompt = ChatPromptTemplate.from_template(
    """
    Exaplain Topic
    """
)

chain = (prompt | llm | StrOutputParser)

results = chain.invoke(
    {
        "topic" : "Apache Spark"
    }
)

# the output will become 
# "Apache Spark is a distributed processing framework..."

#### Understanding the Pipe Operator
- prompt | llm | parser, means input-->prompt-->llm-->parser-->output
- output of one becomes input for next 

### Runnable
- Everything in LCEL is called a Runnable.
Examples 
- Prompt --> ChatpromptTemplate
- LLM --> ChatOpenai
- parser --> stroutputparser
- retriver --> VectorStoreRetriver 
-- All are runnable thus that can be chained.

#### invoke()
- Runs one input    ```chain.invoke({"topic":"kafaka"})``` --> output : "Kafka is a distributed event streaming platform..."

#### batch()
- Runs Many inputs , and this is useful for bulk inputs.
```
chain.batch(
    [
        {"topic":"spark"},
        {"topic":"kafka"},
        {"topic":"Airflow"}
    ]
)
``` 

#### stream()
- returns tokens gradually. same typing effect as a chatgpt
```
for chunk in chain.stream({ "topic":"Spark"}):
    print(chunk)
```

#### ainvoke()
- async excecution, used in fastapi, async apis, agents.
```
result = await chain.ainvoke({"topic":"spark"})
```

#### RunnablePassthrough
- suppose input :  ``` {"question":"what is spark"}
- we need to pass the above as unchanged.
``` from langchain_core_runnables import RunnablePassthrough```
- it simpl forwards the input


#### Runnable with lambda
it allows custom python functions.

In [1]:
# example
from langchain_core.runnables import RunnableLambda

def upper(text):
    return text.upper()

runnable = RunnableLambda(upper)
runnable.invoke("hello")

'HELLO'

#### Runnable Parallel 
- it runs multiple branches simultaneously 
- for example : if we asks ```Explain spark ```
- need 1. summary, 2. Advantages 3.use cases
- it will do the parallel Execution
```
Question
      ↓
-------------------
↓       ↓        ↓
Summary  Pros   Uses
-------------------
```

In [ ]:
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel(
    {
        "summary":summary_chain,
        "advantages":advantage_chain
    }
)

# in ouput we will get like this 
{
 "summary":"...",
 "advantages":"..."
}

#### RunnableSequence
- actully we have chain : prompt | llm | Parser
- the above one creates ````RunnableSequence```
- internally visual : step1-->step2-->step3

#### assign()
- here it will add additional fields 
- suppose we have input ```{""question":"what is spark?"}
- it will assign ```{
    "question": "what is spark?",
    "context": "Apache spark"
}``` it is very usefull in rag

- Example : chain = RunnablePassthrough.assign(context=retriever)


#### LCEL in RAG
- in traditional RAG
```
Question
 ↓
Retriever
 ↓
Context
 ↓
Prompt
 ↓
LLM
 ↓
Parser
```

- in LCEL
```
chain = (
{
 "context": retriever,
 "question": RunnablePassthrough()
}
| prompt
| llm
| StrOutputParser()
)
```


#### Fallback Chains
- suppose GPT fails use cluade, it is very useful in production.
```
chain = (
    gpt_chain.with_fallbacks(
        [claude_chain]
    )
)
```
#### Retry()
- if API fails ```chain.with_retry()``` Automatically retries.


#### Overview
LCEL (LangChain Expression Language) is the modern composition framework in LangChain that allows developers to build chains by connecting reusable runnables such as prompts, chat models, retrievers, and output parsers using the pipe (|) operator. It supports streaming, batch processing, parallel execution, async execution, retries, and complex RAG workflows.



In [ ]:
# Without LCEL — imperative, hard to extend
prompt_value = prompt.format_messages(question=query)
llm_output = llm.invoke(prompt_value)
result = output_parser.parse(llm_output)

# With LCEL — declarative, composable, streaming-ready
chain = prompt | llm | output_parser
result = chain.invoke({"question": query})

In [ ]:
# The Runnable Interface
# Before building chains, understand that every LCEL component shares this interface:
# Every runnable has all four of these
chain.invoke(input)              # single input → single output
chain.stream(input)              # single input → streamed output tokens
chain.batch([input1, input2])    # multiple inputs → multiple outputs (parallel)
await chain.ainvoke(input)       # async single input → output

# Inspect the chain's expected input and output schema
chain.input_schema.schema()
chain.output_schema.schema()

Level 1 — Simplest Chain: Prompt → LLM → Parser


In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
import os 
from dotenv import load_dotenv
load_dotenv()
# Each component
prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer this: {question}"
)
llm = ChatGroq(model=os.getenv("groq_model_name"), temperature=0)
parser = StrOutputParser()

# Wire them with |
chain = prompt | llm | parser
# Run it
result = chain.invoke({"question": "What is LangChain?"})
print(result)


LangChain is an open-source Python library that enables developers to build large language models (LLMs) into production-ready applications. It was created by Lautaro Guisasola, and it's designed to simplify the process of integrating LLMs into various use cases, such as chatbots, virtual assistants, and other conversational AI systems.

LangChain provides a set of tools and abstractions that make it easier to work with LLMs, including:

1. **Model management**: LangChain allows you to easily load, manage, and switch between different LLM models.
2. **Data loading**: It provides a simple way to load and preprocess data for your LLM, including text, images, and other types of data.
3. **Prompt engineering**: LangChain offers a range of tools for crafting effective prompts that elicit the desired responses from your LLM.
4. **Model chaining**: It enables you to chain multiple LLMs together to create complex workflows and applications.
5. **Integration with other libraries**: LangChain is

In [ ]:
# how it works
# prompt   → takes {"question": "..."} → returns ChatPromptValue
#    |
# llm      → takes ChatPromptValue    → returns AIMessage
#    |
# parser   → takes AIMessage          → returns str

Level 2 — Streaming

In [8]:
chain = prompt | llm | parser

# Stream tokens as they arrive
for chunk in chain.stream({"question": "Explain langchain in detail"}):
    print(chunk, end="", flush=True)


# Async streaming (for FastAPI, web apps)
async def stream_response():
    async for chunk in chain.astream({"question": "Explain RAG in detail"}):
        print(chunk, end="", flush=True)

**What is LangChain?**

LangChain is an open-source, Python-based framework for building large language models (LLMs) and multimodal models. It was created by the team at LangChain Labs, a company that aims to make it easier for developers to build and deploy AI models. LangChain provides a set of tools and libraries that enable developers to create complex AI applications, such as chatbots, virtual assistants, and content generators.

**Key Features of LangChain**

1. **Modular Architecture**: LangChain is designed with a modular architecture, which allows developers to easily swap out different components and models. This makes it easy to experiment with different approaches and find the best solution for a particular problem.
2. **LLM Integration**: LangChain provides seamless integration with popular LLMs, such as LLaMA, BERT, and RoBERTa. This makes it easy to leverage the power of these models in your applications.
3. **Multimodal Support**: LangChain supports multimodal models, 

Level 3 — RunnablePassthrough (pass input through unchanged)
- When you need the original input to survive alongside transformed outputs, use RunnablePassthrough.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# Problem: once retriever runs, the original question is lost
# Solution: RunnablePassthrough keeps it alive

retrieval_chain = (
    {
        "context": retriever,              # retriever gets the query string
        "question": RunnablePassthrough()  # original query string passes through
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = retrieval_chain.invoke("What is LangChain?")
# "context" = retrieved documents
# "question" = "What is LangChain?" (unchanged)


RunnablePassthrough.assign() — add new keys without losing existing ones:


In [9]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(
        upper_question=lambda x: x["question"].upper()
    )
    | prompt
    | llm
    | StrOutputParser()
)

chain.invoke({"question": "what is langchain?"})
# Input becomes: {"question": "what is langchain?", "upper_question": "WHAT IS LANGCHAIN?"}

'LangChain is an open-source framework for building large language models (LLMs) and multimodal models. It was created by Joshua Browder, the founder of DoNotPay, a popular AI-powered legal services platform.\n\nLangChain allows developers to build custom LLMs and multimodal models using a variety of tools and libraries, including LLaMA, a large language model developed by Meta AI. The framework provides a set of APIs and tools that enable developers to integrate LLMs into their applications, making it easier to build conversational AI, chatbots, and other AI-powered services.\n\nSome of the key features of LangChain include:\n\n1. **Model integration**: LangChain allows developers to integrate LLMs and multimodal models into their applications, making it easier to build conversational AI and other AI-powered services.\n2. **APIs and tools**: The framework provides a set of APIs and tools that enable developers to build custom LLMs and multimodal models.\n3. **Multimodal support**: Lan

Level 4 — RunnableLambda (any Python function in a chain)
- Wrap any Python function so it behaves as a Runnable in the chain.



In [ ]:
from langchain_core.runnables import RunnableLambda

def format_docs(docs):
    return "\n\n".join(
        f"[Source: {d.metadata.get('source','?')}]\n{d.page_content}"
        for d in docs
    )

def add_word_count(text):
    count = len(text.split())
    return f"{text}\n\n[Word count: {count}]"

chain = (
    {"context": retriever | RunnableLambda(format_docs),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(add_word_count)
)

result = chain.invoke("What is LangChain?")


# Shorthand — Python functions are auto-coerced into RunnableLambda inside | chains:
# These are equivalent
chain = prompt | llm | StrOutputParser() | RunnableLambda(str.upper)
chain = prompt | llm | StrOutputParser() | str.upper   # auto-coerced

Level 5 — RunnableParallel (run branches simultaneously)
- Run multiple chains or operations in parallel, then combine the results.

In [10]:
from langchain_core.runnables import RunnableParallel

# Run two LLM calls simultaneously
parallel_chain = RunnableParallel(
    pros=ChatPromptTemplate.from_template("List pros of {topic}") | llm | StrOutputParser(),
    cons=ChatPromptTemplate.from_template("List cons of {topic}") | llm | StrOutputParser()
)

result = parallel_chain.invoke({"topic": "microservices"})
print(result["pros"])   # pros of microservices
print(result["cons"])   # cons of microservices

# Dict shorthand (exact same thing)
parallel_chain = {
    "pros": ChatPromptTemplate.from_template("List pros of {topic}") | llm | StrOutputParser(),
    "cons": ChatPromptTemplate.from_template("List cons of {topic}") | llm | StrOutputParser()
}

Here are some pros of microservices:

1. **Scalability**: Microservices allow for independent scaling of each service, which means that if one service is experiencing high traffic, it can be scaled up without affecting other services.

2. **Flexibility**: Microservices enable the use of different programming languages, frameworks, and databases for each service, allowing developers to choose the best tool for the job.

3. **Fault Tolerance**: If one service fails, it will not bring down the entire application, as each service is independent and can continue to function.

4. **Easier Maintenance**: With microservices, maintenance and updates can be done independently for each service, reducing the risk of downtime and making it easier to deploy new features.

5. **Improved Agility**: Microservices enable faster development and deployment of new features, as each service can be developed and deployed independently.

6. **Better Resource Utilization**: Microservices allow for more efficie

Level 8 — Branching with RunnableBranch
- Route to different chains based on a condition — like an if/else in your pipeline.


In [11]:
from langchain_core.runnables import RunnableBranch

# Different prompts for different question types
technical_prompt = ChatPromptTemplate.from_template(
    "Answer this technical question with code examples: {question}"
)
general_prompt = ChatPromptTemplate.from_template(
    "Answer this question clearly and simply: {question}"
)
financial_prompt = ChatPromptTemplate.from_template(
    "Answer this financial question with data: {question}"
)

# Classifier chain decides which branch to use
classifier_prompt = ChatPromptTemplate.from_template(
    """Classify this question into exactly one category:
    technical / financial / general
    
    Question: {question}
    Category:"""
)
classifier = classifier_prompt | llm | StrOutputParser()

# Branch: (condition_function, chain_to_run)
branch = RunnableBranch(
    (lambda x: "technical" in x["topic"].lower(), technical_prompt | llm | StrOutputParser()),
    (lambda x: "financial" in x["topic"].lower(), financial_prompt | llm | StrOutputParser()),
    general_prompt | llm | StrOutputParser()   # default branch
)

# Full routing chain
def classify_and_route(input):
    topic = classifier.invoke({"question": input["question"]})
    return {"question": input["question"], "topic": topic}

full_chain = RunnableLambda(classify_and_route) | branch

print(full_chain.invoke({"question": "How does backpropagation work?"}))
print(full_chain.invoke({"question": "What is EBITDA?"}))
print(full_chain.invoke({"question": "Who wrote Hamlet?"}))

**Backpropagation: A Fundamental Algorithm in Neural Networks**

Backpropagation is a widely used algorithm in neural networks for training and optimizing the model's parameters. It's a method for minimizing the error between the model's predictions and the actual output. In this explanation, we'll break down the backpropagation process with code examples using Python and the NumPy library.

**Mathematical Background**
-------------------------

Before diving into the code, let's cover the mathematical background. The backpropagation algorithm involves two main steps:

1.  **Forward Pass**: The input data flows through the network, and the output is calculated using the current weights and biases.
2.  **Backward Pass**: The error between the predicted output and the actual output is calculated, and the weights and biases are updated to minimize the error.

**Forward Pass**
----------------

In the forward pass, the input data flows through the network, and the output is calculated usin

Level 10 — Chain Introspection and Debugging

In [ ]:
# Print the full chain structure
chain.get_graph().print_ascii()
#      +---------------------------+
#      | Parallel<context,question>|
#      +---------------------------+
#             *          *
#             *          *
#        +--------+  +------------------+
#        |Retriever|  |RunnablePassthrough|
#        +--------+  +------------------+
#              ...

# See input/output schemas
print(chain.input_schema.schema())
print(chain.output_schema.schema())

# Verbose mode — see every intermediate value
from langchain.callbacks import StdOutCallbackHandler

result = chain.invoke(
    {"question": "What is LangChain?"},
    config={"callbacks": [StdOutCallbackHandler()]}
)

# Trace individual steps with intermediate outputs
from langchain_core.runnables import RunnablePassthrough

debug_chain = (
    RunnablePassthrough.assign(prompt_output=prompt)
    .assign(llm_output=lambda x: llm.invoke(x["prompt_output"]))
    .assign(final=lambda x: StrOutputParser().invoke(x["llm_output"]))
)
result = debug_chain.invoke({"question": "What is LangChain?"})
print(result["prompt_output"])   # see the formatted prompt
print(result["llm_output"])      # see the raw LLM response
print(result["final"])           # see the parsed output